# 09 - Transformer OASIS MIA (depuis data prep)

Notebook inspire de Transformer_OASIS_final_shadow_latest, mais simplifie et branche sur la sortie de preparation OASIS.

Objectifs:
- Charger prepared_oasis2_transformer.npz
- Entrainer un modele safe et un modele risky
- Evaluer MIA standard et MIA robuste (black-box)
- Afficher un resume clair des AUC

In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, roc_auc_score
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

# Import des protocoles MIA depuis transformer_pipeline
repo_root = os.getcwd()
proto_dir = os.path.join(repo_root, 'transformer_pipeline')
if proto_dir not in sys.path:
    sys.path.append(proto_dir)

from research_protocol import evaluate_mia_research_protocol, set_seed
from research_protocol_robust import evaluate_mia_robust_protocol

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

def set_global_determinism(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

In [2]:
SEED = 42
ATTACK_SEEDS = [11, 22, 33]

# Safe model
TARGET_DROPOUT_SAFE = 0.15
TARGET_EPOCHS_SAFE = 120

# Risky model (plus favorable a MIA)
TARGET_DROPOUT_RISKY = 0.00
TARGET_EPOCHS_RISKY = 240
RISKY_TRAIN_FRACTION = 0.70
RISKY_BATCH_SIZE = 32

# Attaquant
N_SHADOWS = 24
SHADOW_EPOCHS = 60
SHADOW_BATCH = 16

ROBUST_N_SHADOWS = 30
ROBUST_SHADOW_EPOCHS = 60
ROBUST_BOUNDARY_DIRS = 24
ROBUST_BOUNDARY_STEPS = 8
ROBUST_BOUNDARY_MAX_ALPHA = 2.0

ATTACK_MIN_AUC_TARGET = 0.55
RUN_ATTACKS = True

set_global_determinism(SEED)

candidate_paths = [
    os.path.join('data_preparation', 'prepared_oasis2_transformer.npz'),
    os.path.join('..', 'data_preparation', 'prepared_oasis2_transformer.npz'),
]
prepared_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if prepared_path is None:
    raise FileNotFoundError('prepared_oasis2_transformer.npz introuvable. Executer data_preparation/01_data_preparation.ipynb')

b = np.load(prepared_path)
X_train = b['X_train'].astype(np.float32)
X_test = b['X_test'].astype(np.float32)
y_train = b['y_train'].astype(np.int32)
y_test = b['y_test'].astype(np.int32)

if 'X_shadow' in b:
    X_shadow_raw = b['X_shadow'].astype(np.float32)
else:
    X_shadow_raw = b['X_shadow_raw'].astype(np.float32)
y_shadow = b['y_shadow'].astype(np.int32)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1]).astype(np.float32)
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1]).astype(np.float32)

print(f'Prepared file: {prepared_path}')
print(f'Loaded: X_train={X_train_seq.shape}, X_test={X_test_seq.shape}, X_shadow={X_shadow_raw.shape}')
print(f'Class ratio train={y_train.mean():.4f}, test={y_test.mean():.4f}')

def build_transformer(n_features, dropout=0.15, d_model=64, ff_dim=128):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Flatten()(inp)
    x = layers.Dense(d_model, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(ff_dim, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(d_model, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

Prepared file: data_preparation\prepared_oasis2_transformer.npz
Loaded: X_train=(70, 1, 9), X_test=(284, 1, 9), X_shadow=(107, 9)
Class ratio train=0.3571, test=0.3592


In [3]:
print('=== 1) Safe utility model ===')
set_seed(SEED + 1)
tf.keras.backend.clear_session()

model_safe = build_transformer(n_features=X_train.shape[1], dropout=TARGET_DROPOUT_SAFE)
es = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
model_safe.fit(
    X_train_seq,
    y_train,
    epochs=TARGET_EPOCHS_SAFE,
    batch_size=32,
    validation_split=0.2,
    callbacks=[es],
    verbose=0,
)

p_tr_safe = model_safe.predict(X_train_seq, verbose=0).ravel()
p_te_safe = model_safe.predict(X_test_seq, verbose=0).ravel()
safe_train_acc = accuracy_score(y_train, (p_tr_safe >= 0.5).astype(int))
safe_test_acc = accuracy_score(y_test, (p_te_safe >= 0.5).astype(int))
display(pd.DataFrame([{
    'model': 'safe_reference',
    'train_acc': safe_train_acc,
    'test_acc': safe_test_acc,
    'gap': safe_train_acc - safe_test_acc,
    'train_auc': roc_auc_score(y_train, p_tr_safe),
    'test_auc': roc_auc_score(y_test, p_te_safe),
}]).round(4))

print('\n=== 2) Risky target model ===')
set_seed(SEED + 2)
tf.keras.backend.clear_session()

rng = np.random.default_rng(SEED + 2026)
n_risky = int(len(X_train_seq) * RISKY_TRAIN_FRACTION)
risky_idx = rng.choice(len(X_train_seq), size=n_risky, replace=False)
X_train_risky_seq = X_train_seq[risky_idx]
y_train_risky = y_train[risky_idx]
print(f'Risky train subset: {len(X_train_risky_seq)} / {len(X_train_seq)} ({RISKY_TRAIN_FRACTION:.0%})')

model_attack = build_transformer(
    n_features=X_train.shape[1],
    dropout=TARGET_DROPOUT_RISKY,
    d_model=96,
    ff_dim=192,
)
model_attack.fit(
    X_train_risky_seq,
    y_train_risky,
    epochs=TARGET_EPOCHS_RISKY,
    batch_size=RISKY_BATCH_SIZE,
    verbose=0,
)

p_tr_risky_subset = model_attack.predict(X_train_risky_seq, verbose=0).ravel()
p_te_attack = model_attack.predict(X_test_seq, verbose=0).ravel()
attack_train_acc = accuracy_score(y_train_risky, (p_tr_risky_subset >= 0.5).astype(int))
attack_test_acc = accuracy_score(y_test, (p_te_attack >= 0.5).astype(int))
display(pd.DataFrame([{
    'model': 'attack_target',
    'train_acc_subset': attack_train_acc,
    'test_acc': attack_test_acc,
    'gap_subset_minus_test': attack_train_acc - attack_test_acc,
    'train_auc_subset': roc_auc_score(y_train_risky, p_tr_risky_subset),
    'test_auc': roc_auc_score(y_test, p_te_attack),
}]).round(4))

=== 1) Safe utility model ===



,model,train_acc,test_acc,gap,train_auc,test_auc
0,safe_reference,0.9429,0.9155,0.0274,0.9902,0.9874



=== 2) Risky target model ===
Risky train subset: 49 / 70 (70%)


,model,train_acc_subset,test_acc,gap_subset_minus_test,train_auc_subset,test_auc
0,attack_target,1.0,0.9542,0.0458,1.0,0.9933


In [4]:
if not RUN_ATTACKS:
    print('RUN_ATTACKS=False -> attaques sautees')
else:
    print('=== 3) MIA standard ===')
    baseline_summary, baseline_per_seed = evaluate_mia_research_protocol(
        target_model=model_attack,
        X_train_seq=X_train_risky_seq,
        y_train=y_train_risky,
        X_test_seq=X_test_seq,
        y_test=y_test,
        X_shadow_raw=X_shadow_raw,
        y_shadow=y_shadow,
        model_builder=lambda nf: build_transformer(n_features=nf, dropout=TARGET_DROPOUT_RISKY, d_model=96, ff_dim=192),
        seed_list=ATTACK_SEEDS,
        n_shadows=N_SHADOWS,
        shadow_epochs=SHADOW_EPOCHS,
        shadow_batch_size=SHADOW_BATCH,
    )
    display(baseline_summary.round(4))

    print('\n=== 4) MIA robuste (black-box) ===')
    baseline_robust_summary, baseline_robust_per_seed = evaluate_mia_robust_protocol(
        target_model=model_attack,
        x_train_seq=X_train_risky_seq,
        y_train=y_train_risky,
        x_test_seq=X_test_seq,
        y_test=y_test,
        x_shadow_raw=X_shadow_raw,
        y_shadow=y_shadow,
        model_builder=lambda nf: build_transformer(n_features=nf, dropout=TARGET_DROPOUT_RISKY, d_model=96, ff_dim=192),
        seed_list=ATTACK_SEEDS,
        n_shadows=ROBUST_N_SHADOWS,
        shadow_epochs=ROBUST_SHADOW_EPOCHS,
        shadow_batch_size=SHADOW_BATCH,
        boundary_dirs=ROBUST_BOUNDARY_DIRS,
        boundary_steps=ROBUST_BOUNDARY_STEPS,
        boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
    )
    display(baseline_robust_summary.round(4))

    quick_standard = baseline_summary[['attack', 'auc_mean', 'auc_std']].sort_values('auc_mean', ascending=False)
    quick_robust = baseline_robust_per_seed[['seed', 'auc', 'selected_scorer', 'tpr_at_1pct_fpr', 'tpr_at_5pct_fpr']]

    print('\n=== 5) Resume AUC ===')
    display(quick_standard.round(4))
    display(quick_robust.round(4))

    std_auc = float(quick_standard.iloc[0]['auc_mean'])
    rob_auc = float(baseline_robust_summary.iloc[0]['auc_mean'])
    if std_auc < ATTACK_MIN_AUC_TARGET and rob_auc < ATTACK_MIN_AUC_TARGET:
        print(f'WARNING: attaques encore faibles (std={std_auc:.4f}, robust={rob_auc:.4f})')
        print('Augmenter shadows/epochs ou reduire RISKY_TRAIN_FRACTION.')
    else:
        print(f'OK: signal MIA detecte (std={std_auc:.4f}, robust={rob_auc:.4f})')

=== 3) MIA standard ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,tpr_at_1pct_fpr_mean,tpr_at_1pct_fpr_std,tpr_at_5pct_fpr_mean,tpr_at_5pct_fpr_std
1,shadow_meta,0.5488,0.0556,0.3433,0.0075,0.1666,0.0059,0.85,0.05,0.2786,0.0109,0.0167,0.0289,0.0167,0.0289
0,logistic,0.5344,0.0312,0.8507,0.0000,0.0000,0.0000,0.00,0.00,0.0000,0.0000,0.0000,0.0000,0.0333,0.0289
2,threshold_loss,0.5335,0.0299,0.2289,0.0215,0.1622,0.0038,1.00,0.00,0.2791,0.0057,0.0000,0.0000,0.0000,0.0000



=== 4) MIA robuste (black-box) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,shadow_meta,0.5955,0.0986,0.505,0.0524,0.1542,0.0524,0.5167,0.1756,0.2375,0.0807



=== 5) Resume AUC ===


,attack,auc_mean,auc_std
1,shadow_meta,0.5488,0.0556
0,logistic,0.5344,0.0312
2,threshold_loss,0.5335,0.0299


,seed,auc,selected_scorer,tpr_at_1pct_fpr,tpr_at_5pct_fpr
0,11,0.7039,lira_only,0.05,0.20
1,22,0.5114,boundary_only,0.15,0.15
2,33,0.5711,lira_only,0.15,0.20


OK: signal MIA detecte (std=0.5488, robust=0.5955)


## Notes
- Si le notebook est lance depuis un autre dossier, verifier les chemins data_preparation et transformer_pipeline.
- Pour accelerer, reduire N_SHADOWS/ROBUST_N_SHADOWS et SHADOW_EPOCHS.
- Pour rendre l'attaque plus forte, baisser RISKY_TRAIN_FRACTION (ex: 0.60) ou augmenter TARGET_EPOCHS_RISKY.

## Enhanced audit: shadow diversity, gradient-based attack, and behavioral log detection

Cette section ajoute trois renforcements:
- plus de diversité dans les shadow models (hyperparametres varies)
- une attaque gradient-based blanche-boîte sur le modele cible
- un detecteur de sessions/logs comportementaux pour un cas plus realiste

In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

def tpr_at_fpr(y_true, y_score, target_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    idx = np.searchsorted(fpr, target_fpr, side='right') - 1
    idx = int(np.clip(idx, 0, len(tpr) - 1))
    return float(tpr[idx])

def score_summary(name, y_true, y_score):
    y_pred = (y_score >= 0.5).astype(int)
    return {
        'attack': name,
        'auc': roc_auc_score(y_true, y_score),
        'tpr_at_1pct_fpr': tpr_at_fpr(y_true, y_score, 0.01),
        'tpr_at_5pct_fpr': tpr_at_fpr(y_true, y_score, 0.05),
        'accuracy': float((y_pred == y_true).mean()),
    }

class DiverseMLPBuilder:
    def __init__(self, seed=SEED):
        self.rng = np.random.default_rng(seed)
        self.configs = [
            {'dropout': 0.00, 'width1': 64, 'width2': 128, 'lr': 1e-3},
            {'dropout': 0.05, 'width1': 96, 'width2': 160, 'lr': 8e-4},
            {'dropout': 0.10, 'width1': 128, 'width2': 192, 'lr': 7e-4},
            {'dropout': 0.15, 'width1': 160, 'width2': 192, 'lr': 6e-4},
        ]
        self.call_count = 0

    def __call__(self, n_features):
        cfg = self.configs[self.call_count % len(self.configs)]
        self.call_count += 1
        inp = layers.Input(shape=(1, n_features))
        x = layers.Flatten()(inp)
        x = layers.Dense(cfg['width1'], activation='relu')(x)
        x = layers.Dropout(cfg['dropout'])(x)
        x = layers.Dense(cfg['width2'], activation='relu')(x)
        x = layers.Dropout(cfg['dropout'])(x)
        x = layers.Dense(64, activation='relu')(x)
        x = layers.Dropout(cfg['dropout'])(x)
        out = layers.Dense(1, activation='sigmoid')(x)
        model = tf.keras.Model(inp, out)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg['lr']),
            loss='binary_crossentropy',
            metrics=['accuracy'],
        )
        return model

print('=== Enhanced shadow diversity benchmark ===')
diverse_builder = DiverseMLPBuilder(seed=SEED + 77)
diverse_summary, diverse_per_seed = evaluate_mia_research_protocol(
    target_model=model_attack,
    X_train_seq=X_train_risky_seq,
    y_train=y_train_risky,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=diverse_builder,
    seed_list=ATTACK_SEEDS,
    n_shadows=36,
    shadow_epochs=70,
    shadow_batch_size=16,
 )
display(diverse_summary.round(4))

print('=== Gradient-based white-box attack (last-layer gradients) ===')

def last_layer_gradient_scores(model, X_seq, y_true):
    y_true = np.asarray(y_true, dtype=np.float32).reshape(-1, 1)
    scores = []
    for i in range(len(X_seq)):
        x_i = tf.convert_to_tensor(X_seq[i:i + 1], dtype=tf.float32)
        y_i = tf.convert_to_tensor(y_true[i:i + 1], dtype=tf.float32)
        with tf.GradientTape() as tape:
            preds = model(x_i, training=False)
            loss = tf.keras.losses.binary_crossentropy(y_i, preds)
            loss = tf.reduce_mean(loss)
        grads = tape.gradient(loss, model.trainable_variables)
        flat = []
        for g in grads[-2:]:
            if g is not None:
                flat.append(tf.reshape(g, [-1]))
        if flat:
            score = tf.norm(tf.concat(flat, axis=0))
        else:
            score = tf.constant(0.0, dtype=tf.float32)
        scores.append(float(-score.numpy()))
    return np.asarray(scores, dtype=np.float64)

grad_train_score = last_layer_gradient_scores(model_attack, X_train_risky_seq, y_train_risky)
grad_test_score = last_layer_gradient_scores(model_attack, X_test_seq, y_test)

y_grad = np.concatenate([np.ones(len(grad_train_score), dtype=int), np.zeros(len(grad_test_score), dtype=int)])
s_grad = np.concatenate([grad_train_score, grad_test_score])
grad_report = score_summary('gradient_last_layer', y_grad, s_grad)
display(pd.DataFrame([grad_report]).round(4))

print('=== Behavioral log detection (session-level) ===')

def session_features_from_probs(probs, points_flat):
    probs = np.asarray(probs, dtype=np.float32)
    points_flat = np.asarray(points_flat, dtype=np.float32)
    rounded = np.round(points_flat, 3)
    _, uniq_idx = np.unique(rounded, axis=0, return_index=True)
    unique_ratio = float(len(uniq_idx) / max(1, len(points_flat)))
    repeat_ratio = 1.0 - unique_ratio
    p = np.clip(probs, 1e-6, 1 - 1e-6)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    conf = np.maximum(p, 1 - p)
    margin = np.abs(p - 0.5)
    return {
        'n_queries': int(len(probs)),
        'unique_ratio': unique_ratio,
        'repeat_ratio': repeat_ratio,
        'p_mean': float(np.mean(p)),
        'p_std': float(np.std(p)),
        'conf_mean': float(np.mean(conf)),
        'conf_std': float(np.std(conf)),
        'entropy_mean': float(np.mean(ent)),
        'entropy_std': float(np.std(ent)),
        'margin_mean': float(np.mean(margin)),
        'margin_std': float(np.std(margin)),
        'near_boundary_ratio': float(np.mean((p > 0.45) & (p < 0.55))),
    }

rng = np.random.default_rng(SEED + 999)
X_test_flat = X_test_seq.reshape(len(X_test_seq), -1)
benign_sessions = []
attack_sessions = []

for sid in range(160):
    idx = rng.choice(len(X_test_flat), size=24, replace=False)
    pts = X_test_flat[idx]
    probs = model_attack.predict(pts.reshape(-1, 1, pts.shape[1]), verbose=0).ravel()
    benign_sessions.append({**session_features_from_probs(probs, pts), 'label_attack': 0, 'session_id': f'benign_{sid}'})

for sid in range(160):
    anchor_idx = rng.choice(len(X_test_flat), size=4, replace=False)
    anchors = X_test_flat[anchor_idx]
    pts = []
    for a in anchors:
        for _ in range(6):
            pts.append(a + rng.normal(0.0, 0.03, size=a.shape).astype(np.float32))
    pts = np.asarray(pts, dtype=np.float32)
    probs = model_attack.predict(pts.reshape(-1, 1, pts.shape[1]), verbose=0).ravel()
    attack_sessions.append({**session_features_from_probs(probs, pts), 'label_attack': 1, 'session_id': f'attack_{sid}'})

df_logs = pd.DataFrame(benign_sessions + attack_sessions)
log_features = [c for c in df_logs.columns if c not in ['session_id', 'label_attack']]
X_log = df_logs[log_features].astype(np.float32).values
y_log = df_logs['label_attack'].values

Xtr, Xte, ytr, yte = train_test_split(X_log, y_log, test_size=0.30, random_state=SEED, stratify=y_log)
log_clf = LogisticRegression(max_iter=1000, random_state=SEED)
log_clf.fit(Xtr, ytr)
log_score = log_clf.predict_proba(Xte)[:, 1]
log_pred = (log_score >= 0.5).astype(int)

log_report = {
    'attack': 'behavioral_log_detector',
    'auc': roc_auc_score(yte, log_score),
    'tpr_at_1pct_fpr': tpr_at_fpr(yte, log_score, 0.01),
    'tpr_at_5pct_fpr': tpr_at_fpr(yte, log_score, 0.05),
    'accuracy': accuracy_score(yte, log_pred),
}
display(pd.DataFrame([log_report]).round(4))

print('Confusion matrix:')
print(confusion_matrix(yte, log_pred))
print('Classification report:')
print(classification_report(yte, log_pred, digits=4))

print('Top features for session detector:')
coef_df = pd.DataFrame({
    'feature': log_features,
    'coef_abs': np.abs(log_clf.coef_[0]),
    'coef': log_clf.coef_[0],
}).sort_values('coef_abs', ascending=False)
display(coef_df.head(10).round(4))

print('=== Low-FPR comparison ===')
comparison_rows = [
    {**score_summary('gradient_last_layer', y_grad, s_grad)},
    {**log_report},
 ]
display(pd.DataFrame(comparison_rows).round(4))

=== Enhanced shadow diversity benchmark ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,tpr_at_1pct_fpr_mean,tpr_at_1pct_fpr_std,tpr_at_5pct_fpr_mean,tpr_at_5pct_fpr_std
0,logistic,0.5344,0.0312,0.8507,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0333,0.0289
2,threshold_loss,0.5335,0.0299,0.2289,0.0215,0.1622,0.0038,1.0000,0.0000,0.2791,0.0057,0.0,0.0,0.0000,0.0000
1,shadow_meta,0.5136,0.1024,0.5174,0.0777,0.1543,0.0425,0.4833,0.0764,0.2335,0.0570,0.0,0.0,0.0333,0.0577


=== Gradient-based white-box attack (last-layer gradients) ===


,attack,auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,accuracy
0,gradient_last_layer,0.5198,0.0,0.0,0.8529


=== Behavioral log detection (session-level) ===


,attack,auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,accuracy
0,behavioral_log_detector,0.8277,0.5208,0.7292,0.8229


Confusion matrix:
[[43  5]
 [12 36]]
Classification report:
              precision    recall  f1-score   support

           0     0.7818    0.8958    0.8350        48
           1     0.8780    0.7500    0.8090        48

    accuracy                         0.8229        96
   macro avg     0.8299    0.8229    0.8220        96
weighted avg     0.8299    0.8229    0.8220        96

Top features for session detector:


,feature,coef_abs,coef
4,p_std,2.6975,-2.6975
8,entropy_std,2.1589,-2.1589
6,conf_std,1.1535,-1.1535
10,margin_std,1.1535,-1.1535
3,p_mean,0.4913,0.4913
7,entropy_mean,0.2720,0.2720
9,margin_mean,0.0970,-0.0970
5,conf_mean,0.0959,-0.0959
0,n_queries,0.0549,0.0549
11,near_boundary_ratio,0.0543,0.0543


=== Low-FPR comparison ===


,attack,auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,accuracy
0,gradient_last_layer,0.5198,0.0000,0.0000,0.8529
1,behavioral_log_detector,0.8277,0.5208,0.7292,0.8229
